# Rat Detection Model — Train on Colab T4, Use in VS Code

This notebook trains a YOLOv8 model to detect and classify rats/rodents in images and video.

**How to use:**
1. Runtime -> Change runtime type -> GPU -> T4
2. Run cells top to bottom
3. At the end, download `best.pt` and use it in VS Code with your normal PC (no GPU needed for just running it)


## 1. Check GPU

In [ ]:
!nvidia-smi

## 2. Install packages

In [ ]:
!pip install ultralytics roboflow -q

## 3. Get a rat/rodent dataset

We use Roboflow Universe, it has ready-made rat detection datasets.

1. Go to https://universe.roboflow.com and search "rat detection" or "rodent detection"
2. Pick a dataset, click "Download this Dataset" -> format **YOLOv8** -> copy the code snippet
3. It gives you an API key + workspace + project + version, paste them below

If you already have your own dataset in YOLO format, skip this and just upload it to `/content/dataset` with a `data.yaml` inside.


In [ ]:
from roboflow import Roboflow

# paste your own values here, get them from roboflow universe dataset page
ROBOFLOW_API_KEY = "YOUR_API_KEY"
WORKSPACE = "YOUR_WORKSPACE"
PROJECT = "YOUR_PROJECT"
VERSION = 1

rf = Roboflow(api_key=ROBOFLOW_API_KEY)
project = rf.workspace(WORKSPACE).project(PROJECT)
dataset = project.version(VERSION).download("yolov8")

print(dataset.location)

## 4. Train the model (uses Colab's T4 GPU)

In [ ]:
from ultralytics import YOLO

model = YOLO("yolov8n.pt")  # nano, fast and light, good for T4 and later for your PC

results = model.train(
    data=f"{dataset.location}/data.yaml",
    epochs=60,
    imgsz=640,
    batch=16,
    device=0,
    project="rat_detector",
    name="train1"
)

## 5. Validate the model

In [ ]:
metrics = model.val()
print(metrics)

## 6. Test on 2-3 images

Upload a few rat images to `/content/test_images/` (use the folder icon on the left, or the upload cell below), then run.


In [ ]:
from google.colab import files
import os

os.makedirs("/content/test_images", exist_ok=True)
uploaded = files.upload()  # pick 2-3 rat images from your PC

for name in uploaded.keys():
    os.rename(name, f"/content/test_images/{name}")

In [ ]:
best_model = YOLO("rat_detector/train1/weights/best.pt")

test_results = best_model.predict(
    source="/content/test_images",
    conf=0.4,
    save=True
)

print("check results in runs/detect/predict or rat_detector/train1/predict")

In [ ]:
import glob
from IPython.display import Image, display

for img_path in glob.glob("runs/detect/predict/*.jpg"):
    display(Image(filename=img_path))

## 7. Test on a 5-10 second video

Upload a short rat video clip below.


In [ ]:
uploaded_video = files.upload()  # pick your 5-10 sec test video
video_name = list(uploaded_video.keys())[0]

In [ ]:
video_results = best_model.predict(
    source=video_name,
    conf=0.4,
    save=True
)

print("processed video saved in runs/detect/predict (an mp4 file)")

## 8. Download your trained model

This is the file you need. Download `best.pt` and put it in your VS Code project folder.
Your PC does NOT need a GPU to just run this model, CPU is fine for normal use.


In [ ]:
from google.colab import files

files.download("rat_detector/train1/weights/best.pt")

## 9. How to use `best.pt` in VS Code (just for reference)

Install once: `pip install ultralytics`

```python
from ultralytics import YOLO

model = YOLO("best.pt")

# image
model.predict(source="test.jpg", show=True, save=True)

# webcam / real time
model.predict(source=0, show=True)

# video file
model.predict(source="test_video.mp4", show=True, save=True)
```

That's it, no GPU needed on your PC, `ultralytics` will just use CPU automatically.
